# Exploración del Titanic

Análisis exploratorio del taller de Buenas Prácticas en Ciencia de Datos e IA.

Este cuaderno lee la capa **Silver** que produce `flows/01_data_engineering.py`,
nunca la fuente original. Para reproducirlo desde cero:

In [ ]:
# 1) Importamos las librerías con sus alias convencionales
import pandas as pd                 # tablas (DataFrame)
import numpy as np                  # cálculo numérico y condiciones vectorizadas
import matplotlib.pyplot as plt     # gráficos base (Seaborn se apoya en él)
import seaborn as sns               # gráficos estadísticos de alto nivel
from scipy import stats             # pruebas estadísticas (Parte 6)

from flows.common.paths import BRONZE, SILVER, GOLD, BRONZE_LABELED_NAME, SILVER_NAME

# 2) Estilo global: todos los gráficos heredan esta configuración
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)   # tamaño por defecto (ancho, alto) en pulgadas
plt.rcParams['axes.titlesize'] = 13

# 3) A partir de la Parte 2 el cuaderno lee Silver, no Bronze:
#    consume lo que produce el pipeline, no el CSV crudo.
df = pd.read_parquet(SILVER / SILVER_NAME)

## Parte 1 — Primer contacto con los datos

### Pasos 1 a 4. Forma, tipos y estadísticos

Cuántas filas y columnas hay, de qué tipo es cada una, y cómo se distribuyen los valores numéricos y los categóricos.

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()

### Paso 5. Datos faltantes

Dónde faltan datos, cuánto falta en cada columna, y si los huecos siguen un patrón o están repartidos al azar.

In [ ]:
# Conteo de valores faltantes por columna
df.isnull().sum()

In [ ]:
# Solo las columnas incompletas, con su porcentaje, de mayor a menor
faltantes = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "pct": (df.isnull().sum() / len(df) * 100).round(2),
})
faltantes[faltantes["nulos"] > 0].sort_values("nulos", ascending=False)

In [ ]:
# Mapa de datos faltantes: cada franja clara es un valor ausente.
# Permite ver si los huecos siguen un patrón o están repartidos al azar.
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Datos faltantes en Silver (891 pasajeros)')
plt.show()

In [ ]:
# ¿Falta Cabin por igual en todas las clases? Proporción de faltantes por clase
pd.crosstab(df['Pclass'], df['Cabin'].isnull(), normalize='index').round(3)

**Lectura del cruce.** Los faltantes de `Cabin` no son aleatorios: falta en el 18.5 % de primera clase y en el 97.6 % de tercera.

El dato ausente **no significa que el pasajero no tuviera camarote**: significa que no quedó registrado. Lo más probable es que refleje cómo se compilaron los manifiestos, con más detalle para primera clase, y no una diferencia física en el alojamiento. Es una asociación, no una causa.

Consecuencia para el modelado: `Cabin` no se imputa. Y una variable binaria "tiene camarote" sería casi un duplicado de `Pclass`.

## Parte 2 — Conociendo a los pasajeros

### Paso 6. Distribución por género y su cruce con la clase

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase')
plt.show()

In [ ]:
# Normalizado POR COLUMNAS: cada clase suma 1, así se comparan
# clases de tamaño distinto sin que la más grande domine.
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)

### Paso 7. Distribución de edades

`Age` todavía **no está imputada** en Silver (eso llega en el Paso 22), así que aquí solo se ven los 714 pasajeros con edad conocida.

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30)
plt.title('Distribución de edades (solo pasajeros con edad conocida)')
plt.xlabel('Edad')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# Pandas ignora los NaN automáticamente en estas agregaciones
df['Age'].agg(['mean', 'median', 'min', 'max']).round(2)

### Paso 8. La variable `Person`

`Person` la crea `add_person()` en `flows/01_data_engineering.py`, no el cuaderno:
`'child'` si `Age < 16`, y el valor de `Sex` en caso contrario.

**Regla explícita para edad desconocida:** en NumPy, `NaN < 16` es `False`, así que un pasajero sin edad nunca se marca como `child`. Eso subestima a los niños sin edad registrada, y el Reto 8.1 cuantifica cuánto.

In [ ]:
df['Person'].value_counts()

In [ ]:
# Reto 8.1: ¿cuántos 'child' se pierden por falta de edad registrada?
sin_edad = int(df['Age'].isna().sum())
p_ninos = (df.loc[df['Age'].notna(), 'Age'] < 16).mean()
print(f"Pasajeros sin edad registrada: {sin_edad}")
print(f"Menores de 16 entre los que sí tienen edad: {p_ninos:.1%}")
print(f"Estimación de 'child' no contados: ~{round(sin_edad * p_ninos)}")

### Paso 9. ¿Cuántos niños viajaban en cada clase?

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

### Paso 10. Densidades de edad (KDE)

Un KDE es un histograma suavizado: la curva muestra dónde se concentran los
valores. `common_norm=False` normaliza **cada grupo por separado**; sin eso, los
grupos grandes aplastan visualmente a los pequeños y el gráfico engaña.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

sns.kdeplot(data=df, x='Age', hue='Sex', fill=True, common_norm=False, ax=axes[0])
axes[0].set_title('Edad por sexo')

sns.kdeplot(data=df, x='Age', hue='Person', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('Edad por tipo de pasajero')

sns.kdeplot(data=df, x='Age', hue='Pclass', fill=True, common_norm=False,
            palette='viridis', ax=axes[2])
axes[2].set_title('Edad por clase')

plt.tight_layout()
plt.show()

## Parte 3 — Ubicación, embarque y composición familiar

### Paso 11. La cubierta del barco

`add_cabin_features()` extrae dos columnas de `Cabin`: `TCabin` con la letra de
la cubierta (`'C85'` → `'C'`, y `'X'` cuando no hay cabina registrada), y
`HasCabin` con 0/1.

Separarlas importa porque el Paso 5 mostró que **la ausencia del dato no es
aleatoria**: `HasCabin` captura ese hecho, `TCabin` captura el contenido.

In [ ]:
df['TCabin'].value_counts().sort_index()

### Paso 12. Conteo por cubierta, con depuración de anómalos

In [ ]:
# Solo las cabinas conocidas: la 'X' no es una cubierta, es la ausencia del dato
cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']

sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros por cubierta (solo cabinas registradas)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# ¿Hay alguna cubierta con un solo registro? Se detecta, no se asume.
conteo = cubierta.value_counts()
print(conteo.sort_index())
print(f"\nCubiertas con un solo pasajero: {conteo[conteo == 1].index.tolist()}")

Una cubierta con un solo pasajero **no es un dato erróneo**: ese pasajero
existió y su cabina está registrada. Es un problema de estadística, no de
calidad del dato — un grupo de tamaño 1 no permite estimar nada y distorsiona
la escala del gráfico.

Por eso se excluye **solo del gráfico** y la fila se conserva intacta en Silver.

In [ ]:
cubierta_sin_anomala = cubierta[cubierta != 'T']

sns.countplot(x=cubierta_sin_anomala, order=sorted(cubierta_sin_anomala.unique()))
plt.title('Pasajeros por cubierta (sin la cubierta T, de un solo registro)')
plt.xlabel('Cubierta')
plt.ylabel('Pasajeros')
plt.show()

### Paso 13. Puerto de embarque y clase

`S` = Southampton, `C` = Cherburgo, `Q` = Queenstown.

In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass')
plt.title('Puerto de embarque por clase')
plt.xlabel('Puerto')
plt.ylabel('Pasajeros')
plt.show()

In [ ]:
# Normalizado por FILAS: cada puerto suma 1.
# Responde "¿qué composición de clases tenía cada puerto?",
# no "¿de dónde venía cada clase?" — son preguntas distintas.
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(3)

### Paso 14. Viajar solo o acompañado

`add_family_features()` calcula `Familia = SibSp + Parch` (familiares a bordo,
sin contar al pasajero) y `Alone` como su versión categórica.

In [ ]:
df['Alone'].value_counts(normalize=True).round(3)

In [ ]:
# El tamaño del grupo, que 'Alone' pierde al reducir todo a dos categorías
df['Familia'].value_counts().sort_index()

### Paso 15. Distribución de `Alone`

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('Pasajeros que viajaban solos frente a los que iban con familia')
plt.xlabel('')
plt.ylabel('Pasajeros')
plt.show()